In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

print(type(X_train))

In [ ]:
# 2. Create TensorDataset objects
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

print(train_dataset[0][0].shape, test_dataset[0][0].shape)
# torch.Size([3, 36, 36]) torch.Size([3, 36, 36])
# oh now i know that there is 3 channels and 36 width 36 height :)

In [ ]:
# 3. Create DataLoaders
# DataLoader for training data
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

# DataLoader for test data
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [ ]:
# 4. Print shape of one batch

X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")

In [ ]:
# 5. Display sample images
images, labels = next(iter(train_loader))

import matplotlib.pyplot as plt
# Convert from (C, H, W) to (H, W, C) for matplotlib
img = images[0].permute(1, 2, 0)
plt.imshow(img)
plt.show()

In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn

class NN4Layer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(NN4Layer, self).__init__()
        self.layer1 = nn.Linear(input_dim, 256)
        self.layer2 = nn.Linear(256, 128)
        self.layer3 = nn.Linear(128, 64)
        self.layer4 = nn.Linear(64, output_dim) # output here will be 1 because it's regression problem
        self.relu = nn.ReLU() # Activation function

    def forward(self, x):
        a1 = self.relu(self.layer1(x))
        a2 = self.relu(self.layer2(a1))
        a3 = self.relu(self.layer3(a2))
        # Output layer (raw scores)
        output = self.layer4(a3)
        return output

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
  # Set the model to training mode
  model.train()
  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    X_batch = X_batch.flatten(1).to(device) # reshaping the X because it's ([32, 3, 36, 36]) and the model expects it to be 1 dim so ([32, 36*36*3])
    y_batch = y_batch.view(-1, 1).to(device) # instead of (32,) it will be (32, 1) same actually but model's orders ..

    # Forward pass - get model predictions
    outputs = model(X_batch)
    loss = criterion(outputs, y_batch)

    # Backward pass & optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  # Calculate average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
  model.eval()
  running_loss = 0.0

  # Disable gradient computation using torch.no_grad()
  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      X_batch = X_batch.flatten(1).to(device) # reshaping the X because it's ([32, 3, 36, 36]) and the model expects it to be 1 dim so ([32, 36*36*3])
      y_batch = y_batch.view(-1, 1).to(device) # instead of (32,) it will be (32, 1) same actually but model's orders ..

      # Forward pass - get model predictions
      outputs = model(X_batch)

      # Compute loss using criterion
      loss = criterion(outputs, y_batch)

      running_loss += loss.item()

  avg_loss = running_loss / len(test_loader)

  return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NN4Layer(36*36*3, 1).to(device) # Model has to be on device with same data are on --- input_dim = 36*36 - output_dim = 1 NN4Layer(input_dim, output_dim)

Learning_rate = 0.001
optimizer = AdamW(model.parameters(), Learning_rate)
criterion = nn.MSELoss()

In [ ]:
# Task 5: Start training for 20 epochs:
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(20):
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)
  val_loss = validate(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f'Epoch [{epoch+1}/{20}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:
# Plotting results

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: